In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent

sys.path.append(str(PROJECT_ROOT / "src" / "05_build_pytorch_dataset"))

from vad_dataset import VADStackedFrameDataset

In [2]:
train_dataset = VADStackedFrameDataset(
    generated_dir=str(PROJECT_ROOT / "data" / "generated" / "train"),
    split="train",
    manifest_type="noisy"
)

dev_dataset = VADStackedFrameDataset(
    generated_dir=str(PROJECT_ROOT / "data" / "generated" / "dev"),
    split="dev",
    manifest_type="noisy"
)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=512, shuffle=False)

[train] Loaded 3000 examples
[train] Total frames: 6726129
[dev] Loaded 500 examples
[dev] Total frames: 1088453


In [3]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim=1331):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)

In [4]:
sys.path.append(str(PROJECT_ROOT / "src" / "06_train_baseline_mlp"))
from train_utils import train_one_epoch, evaluate

In [5]:
# training loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LogisticRegression().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 3

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}")

    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
    dev_metrics = evaluate(model, dev_loader, criterion, device)

    print("Train F1:", train_metrics["f1"])
    print("Dev F1:", dev_metrics["f1"])


Epoch 1
Train F1: 0.8476619307906754
Dev F1: 0.8343320601433428

Epoch 2
Train F1: 0.8494347711545189
Dev F1: 0.8343467092362798

Epoch 3
Train F1: 0.8494946964979877
Dev F1: 0.8349227445604863


### Logistic Regression Baseline (Reference)

To validate the effectiveness of the MLP model, we trained a logistic regression classifier using the same 1331-dimensional stacked features.

**Configuration**
- Model: Logistic Regression (linear classifier)
- Input dimension: 1331
- Optimizer: Adam
- Learning rate: 0.001
- Batch size: 512
- Epochs: 3

**Results**

| Epoch | Train F1 | Dev F1 |
|------|---------|--------|
| 1 | 0.8477 | 0.8343 |
| 2 | 0.8494 | 0.8343 |
| 3 | 0.8495 | 0.8349 |

**Observations**
- The model converges very quickly (within 2–3 epochs)
- No significant overfitting is observed
- Performance plateaus early, indicating limited model capacity

**Comparison with MLP**
- Logistic Regression Dev F1: ~0.835
- MLP Dev F1: ~0.880

The MLP outperforms logistic regression by approximately **4.5% F1**, demonstrating the benefit of nonlinear modeling.

**Conclusion**
This confirms that the VAD task requires modeling complex feature interactions that cannot be captured by a linear decision boundary.